**Partie I - prediction**

L'objectif est de développer et de valider un modèle de prédiction basé sur l'algorithme des K-Plus-Proches-Voisins (KNN), afin d'anticiper la survenue d'exacerbations chez les patients asthmatiques.

In [18]:

import pandas as pd
import numpy as np
import warnings   #masquer les alertes (warnings) d'affichage dans la console pour que le rendu final soit propre.

#Construction de la Pipeline
from sklearn.pipeline import Pipeline              #Pour enchaîner les traitements proprement (Nettoyage -> Transformation -> Algorithme) sans risque de fuite de données.
from sklearn.compose import ColumnTransformer      #Pour appliquer des transformations différentes selon les colonnes (ex: scaler les nombres, mais pas les catégories).

#Nettoyage et préparation des données
from sklearn.impute import SimpleImputer           #Comble les valeurs manquantes en les remplaçant par la médiane ou la valeur la plus fréquente.
from sklearn.preprocessing import StandardScaler   #Met toutes les valeurs à la même échelle.
from sklearn.preprocessing import OneHotEncoder    #Transforme les variables catégorielles en nouvelles colonnes binaires (0 ou 1).

#L'Algorithme
from sklearn.neighbors import KNeighborsClassifier #Notre modèle, l'algorithme des K-Plus-Proches-Voisins (KNN).

#Outils de découpage et d'évaluation
from sklearn.model_selection import train_test_split #Fonction qui permet de découper notre base de données (Train / Validation / Test).
from sklearn.metrics import f1_score, precision_score #Formules mathématiques pour calculer la performance de notre modèle (Le F1-Score et la Précision).

#Sélection des caractéristiques
from sklearn.feature_selection import SelectKBest, f_classif # Le filtre qui supprime le "bruit" en ne gardant que les 'K' meilleures variables. Empêche le KNN de se perdre.

ÉTAPE 1 : CHARGEMENT ET PRÉPARATION DES DONNÉES

La variable cible doit être binarisée (1 s'il y a eu des exacerbations, 0 sinon).

In [19]:

fichier_entree = 'trainDataDC2.csv.gz'
fichier_sortie = 'predictions.csv'

# Chargement avec gestion stricte des valeurs manquantes ('NA', '', '?', 'None')
df = pd.read_csv(fichier_entree, na_values=['NA', '', '?', 'None'])

# Binarisation de la cible : 0 = sain, 1 = au moins une exacerbation dans l'année
y = (df['post_index_exacerbations365'] > 0).astype(int)
identifiants = df['patid']

# Suppression des variables inutiles (identifiant) ou "du futur" (fuite de données)
X = df.drop(columns=['post_index_exacerbations365', 'patid', 'log_charges', 'log_asthma_charge'])

ÉTAPE 2

- TEST : Le juge final (20% des données intactes).
- TRAIN : Pour entraîner l'algorithme (avec des clones pour équilibrer).
- VALIDATION : Pour choisir notre seuil de décision (doit être sans clones).


In [20]:

#On isole 20% pour le TEST final
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

#On divise le reste en Train et Validation
X_train_sub, X_val, y_train_sub, y_val = train_test_split(
    X_train_full, y_train_full, test_size=0.2, stratify=y_train_full, random_state=42
)

ÉTAPE 3


L'algorithme KNN fait voter les voisins. Si 90% des patients de la base sont sains, le voisinage sera toujours majoritairement sain.

Pour forcer le KNN à détecter la maladie, on clone les malades dans le Train.


In [21]:
df_train_sub = X_train_sub.copy()
df_train_sub['CIBLE'] = y_train_sub

sains = df_train_sub[df_train_sub['CIBLE'] == 0]
malades = df_train_sub[df_train_sub['CIBLE'] == 1]

#Clonage aléatoire des malades jusqu'à atteindre le même nombre que les sains
malades_clones = malades.sample(n=len(sains), replace=True, random_state=42)

#Rassemblement et mélange des données équilibrées
df_train_equilibre = pd.concat([sains, malades_clones]).sample(frac=1, random_state=42)
X_train_bal = df_train_equilibre.drop(columns=['CIBLE'])
y_train_bal = df_train_equilibre['CIBLE']

 ÉTAPE 4

 CRÉATION DU PIPELINE ET PARAMÉTRAGE DU KNN





In [22]:

print("Création du pipeline et entraînement du modèle KNN")

# Catégorisation des variables selon le cahier des charges
categorical_features = [
    'previous_asthma_drugs', 'pneumonia', 'sinusitis', 'acute_bronchitis',
    'acute_laryngitis', 'upper_respiratory_infection', 'gerd', 'rhinitis',
    'drug_s', 'female'
]
numeric_features_int = ['index_age', 'total_pre_index_cannisters_365', 'pre_asthma_days', 'pre_asthma_charge']
numeric_features_float = ['adherence', 'total_pre_index_charge', 'pre_asthma_pharma_charge']

#KNN (StandardScaler) : Le KNN calcule des distances (en mètres, euros, années...).
#Si on ne met pas tout à la même échelle, une variable en milliers (Frais) va totalement écraser une variable en dizaines (Âge). La standardisation est vitale pour le KNN.

num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num_int', num_transformer, numeric_features_int),
    ('num_float', num_transformer, numeric_features_float),
    ('cat', cat_transformer, categorical_features),
])

#PARAMÈTRES DU PIPELINE :
#SelectKBest (k=10) : Le KNN trop de variables = le calcul de distance perd son sens).
#On force le KNN à regarder que les 10 meilleures variables.
#n_neighbors=75 : Un grand voisinage évite le surentraînement.
#p=2 : Distance Euclidienne classique.
# weights='uniform' :tous les voisins ont le même poids de vote. (indispensable avec des clones).
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('feature_selection', SelectKBest(score_func=f_classif, k=10)),
    ('classifier', KNeighborsClassifier(n_neighbors=75, weights='uniform', p=2))
])

# Entraînement effectif sur les données contenant les clones
pipeline.fit(X_train_bal, y_train_bal)

Création du pipeline et entraînement du modèle KNN


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num_int',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['index_age',
                                                   'total_pre_index_cannisters_365',
                                                   'pre_asthma_days',
                                                   'pre_asthma_charge']),
                                                 ('num_float',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler...
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['previous_asthma_drugs',
                                                   'pneumonia', 'sinusitis',
                                                   'acute_bronchitis',
                                                   'acute_laryngitis',
                                                   'upper_respiratory_infection',
                                                   'gerd', 'rhinitis', 'drug_s',
                                                   'female'])])),
                ('feature_selection', SelectKBest()),
                ('classifier', KNeighborsClassifier(n_neighbors=75))])

ÉTAPE 5 : SEUIL OPTIMAL DE DÉCISION

Par défaut, sklearn prédit "1" si > 50% des voisins votent "1".
On teste plusieurs seuils sur le set de Validation sans clones pour maximiser le F1-score.



In [23]:
probas_val = pipeline.predict_proba(X_val)[:, 1]

meilleur_f1_val = 0
meilleur_seuil = 0.50

for seuil in np.arange(0.3, 0.8, 0.02):
    preds_val = (probas_val >= seuil).astype(int)
    score_f1 = f1_score(y_val, preds_val)
    if score_f1 > meilleur_f1_val:
        meilleur_f1_val = score_f1
        meilleur_seuil = seuil

ÉTAPE 6 : ÉVALUATION FINALE SUR LE TEST

Évaluation des performances sur le Test (20% invisibles)

In [24]:
probas_test = pipeline.predict_proba(X_test)[:, 1]
predictions_test = (probas_test >= meilleur_seuil).astype(int)

f1_test = f1_score(y_test, predictions_test)
precision_test = precision_score(y_test, predictions_test, zero_division=0)

print("\n" + "-"*50)
print(" RÉSULTATS SUR L'ÉCHANTILLON DE TEST (20%) ")
print("-"*50)
print(f"Seuil optimal appliqué  : {meilleur_seuil:.2f}")
print(f"Score F1 Validé         : {f1_test:.4f}")
print(f"Précision Validée       : {precision_test:.4f}")
print("-"*50 + "\n")


--------------------------------------------------
 RÉSULTATS SUR L'ÉCHANTILLON DE TEST (20%) 
--------------------------------------------------
Seuil optimal appliqué  : 0.34
Score F1 Validé         : 0.2055
Précision Validée       : 0.1169
--------------------------------------------------



ÉTAPE 7 : ENTRAÎNEMENT FINAL

Pour rendre le meilleur fichier prédictif possible, on réentraîne notre KNN sur l'intégralité des données disponibles (X_train + X_test).



In [25]:
print("Génération du modèle sur 100% des données et export CSV")

df_all = X.copy()
df_all['CIBLE'] = y
sains_all = df_all[df_all['CIBLE'] == 0]
malades_all = df_all[df_all['CIBLE'] == 1]

#Oversampling sur 100% du dataset
malades_all_clones = malades_all.sample(n=len(sains_all), replace=True, random_state=42)
df_all_bal = pd.concat([sains_all, malades_all_clones]).sample(frac=1, random_state=42)

#Entraînement final
pipeline.fit(df_all_bal.drop(columns=['CIBLE']), df_all_bal['CIBLE'])

#Prédiction sur tous les patients (fichier original, donc 1 seule prédiction par patient unique)
probas_finales = pipeline.predict_proba(X)[:, 1]
predictions_finales = (probas_finales >= meilleur_seuil).astype(int)

#Création du DataFrame final et export
df_resultat = pd.DataFrame({
    'patid': identifiants,
    'prediction': predictions_finales
})
df_resultat.to_csv(fichier_sortie, index=False)

print(f"'{fichier_sortie}'")


Génération du modèle sur 100% des données et export CSV
'predictions.csv'


Nous avons appliqué l'algorithme KNN, qui évalue un patient en le comparant à des profils similaires.

Concernant nos performances, nous obtenons un F1-Score d'environ 0,20. Cela peut sembler faible, mais c'est un résultat tout à fait classique et réaliste en médecine. On ne peut pas prédire l'asthme à 100%. De plus une grande partie de nos patients n'ont fait aucune crise. Ce score de 0,20 prouve que notre modèle réussit son objectif principal : isoler une population à risque.